# 财务舞弊数据读取与训练 Notebook

这个 Notebook 展示如何：

- 使用 `pandas.read_excel(..., skiprows=2)` 读取原始表格；
- 将 `Stkcd` 与 `Accper` 设为面板索引；
- 区分特征列与 6 个多标签列 `Vio, V1, ..., V5`；
- 调用项目中的财务舞弊 pipeline 完成 5-fold 的 train/validation/test 切分与模型训练；
- 重点评估主标签 `Vio` 的 F1 Score 和 AUC。


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATA_PATH = PROJECT_ROOT / 'data' / 'fruade_data' / 'Summary Table-sent-v1（空值取最大值、最小值或者保留；其填充0，“内部控制信息披露指数”补充2024年，填充2025年；“内部控制指数”填充2025年）.xlsx'
LABEL_COLUMNS = ['Vio', 'V1', 'V2', 'V3', 'V4', 'V5']
PRIMARY_LABEL = 'Vio'
DATA_PATH


PosixPath('/data/wk/code/github/automl_palyground/data/fruade_data/Summary Table-sent-v1（空值取最大值、最小值或者保留；其填充0，“内部控制信息披露指数”补充2024年，填充2025年；“内部控制指数”填充2025年）.xlsx')

In [2]:
# 直接用 pandas 读取 Excel，并跳过前两行说明。
raw_df = pd.read_excel(DATA_PATH, skiprows=2)
print(f'原始数据形状: {raw_df.shape}')
print('前 12 列:')
print(raw_df.columns[:12].tolist())
raw_df.head()


原始数据形状: (66859, 162)
前 12 列:
['Stkcd', 'ShortName', 'Accper', 'IndustryCode', 'Vio', 'V1', 'V2', 'V3', 'V4', 'V5', 'Ind1_OCF_to_NetProfit', 'Ind2_OCF_to_OpRev']


,Stkcd,ShortName,Accper,IndustryCode,Vio,V1,V2,V3,V4,V5,...,Ind143_AuditCommitteeMeetings,Ind144_AuditFirmMerge_2Yr,Ind145_RegFinIssueCnt_5Yr,Ind146_LegalEnv_Index,Ind147_LitigationCount,Ind148_FraudCount_5Yr,Ind149_PenaltyCount_5Yr,Ind150_ExecMisconduct_5Yr,Ind151_StaffMisconduct_5Yr,Ind152_InfoEnv_Index
0,2,万科A,2006-12-31,K70,0,0,0,0,0,0,...,0,0,0,8.002,1,0,0,0,0,17
1,2,万科A,2007-12-31,K70,0,0,0,0,0,0,...,0,0,0,8.068,0,0,0,0,0,19
2,2,万科A,2008-12-31,K70,0,0,0,0,0,0,...,8,0,0,8.159,0,0,0,0,0,49
3,2,万科A,2009-12-31,K70,0,0,0,0,0,0,...,6,0,0,7.344,0,0,0,0,0,64
4,2,万科A,2010-12-31,K70,0,0,0,0,0,0,...,0,0,0,7.713,0,0,0,0,0,62


In [3]:
# 按你的要求，将 <Stkcd, Accper> 作为面板索引；ShortName 作为公司名称留在元数据中。
panel_df = raw_df.copy()
panel_df['Accper'] = pd.to_datetime(panel_df['Accper'])
panel_df = panel_df.sort_values(['Stkcd', 'Accper']).set_index(['Stkcd', 'Accper'])

label_df = panel_df[LABEL_COLUMNS].copy()
feature_df = panel_df.drop(columns=['ShortName'] + LABEL_COLUMNS)

print(f'特征矩阵形状: {feature_df.shape}')
print(f'标签矩阵形状: {label_df.shape}')
print('标签正样本数量:')
label_df.sum().to_frame('positive_count')


特征矩阵形状: (66859, 153)
标签矩阵形状: (66859, 6)
标签正样本数量:


,positive_count
Vio,7838
V1,1001
V2,154
V3,5418
V4,4357
V5,207


In [5]:
feature_df.columns

Index(['IndustryCode', 'Ind1_OCF_to_NetProfit', 'Ind2_OCF_to_OpRev',
       'Ind3_OCF_to_OpProfit', 'Ind4_CashOpsIndex', 'Ind5_AssetLiabRatio',
       'Ind6_DebtEquityRatio', 'Ind7_EquityMultiplier', 'Ind8_IntCovRatio',
       'Ind9_CurrentRatio',
       ...
       'Ind143_AuditCommitteeMeetings', 'Ind144_AuditFirmMerge_2Yr',
       'Ind145_RegFinIssueCnt_5Yr', 'Ind146_LegalEnv_Index',
       'Ind147_LitigationCount', 'Ind148_FraudCount_5Yr',
       'Ind149_PenaltyCount_5Yr', 'Ind150_ExecMisconduct_5Yr',
       'Ind151_StaffMisconduct_5Yr', 'Ind152_InfoEnv_Index'],
      dtype='object', length=153)

In [6]:
from financial_fraud import (
    create_fraud_model,
    evaluate_fraud_model,
    load_financial_fraud_excel,
    split_financial_fraud_data,
)

# 项目内置 pipeline 会自动做：
# 1. 公司内按时间正向填充；
# 2. 数值特征分位数裁剪、缺失值填补和稳健缩放；
# 3. 类别特征众数填补和独热编码；
# 4. 使用 5-fold 分层切分，并在当前 fold 内保持 train/validation/test=70%/10%/20%。
features, labels, metadata = load_financial_fraud_excel(
    str(DATA_PATH),
    skiprows=2,
    label_columns=LABEL_COLUMNS,
)
dataset_split = split_financial_fraud_data(
    features=features,
    labels=labels,
    metadata=metadata,
    primary_label=PRIMARY_LABEL,
    n_splits=5,
    fold_index=0,
    train_ratio=0.7,
    valid_ratio=0.1,
    test_ratio=0.2,
)

print('fold:', f"{dataset_split.fold_index + 1}/{dataset_split.n_splits}")
print('X_train:', dataset_split.X_train.shape, 'X_valid:', dataset_split.X_valid.shape, 'X_test:', dataset_split.X_test.shape)
print('Vio 训练集正样本率:', round(dataset_split.y_train[PRIMARY_LABEL].mean(), 4))
print('Vio 验证集正样本率:', round(dataset_split.y_valid[PRIMARY_LABEL].mean(), 4))
print('Vio 测试集正样本率:', round(dataset_split.y_test[PRIMARY_LABEL].mean(), 4))


fold: 1/5
X_train: (46801, 155) X_valid: (6686, 155) X_test: (13372, 155)
Vio 训练集正样本率: 0.1172
Vio 验证集正样本率: 0.1173
Vio 测试集正样本率: 0.1173


In [7]:
# LightGBM 是这个任务的推荐起点；如果环境里没有 lightgbm，会自动回退到随机森林。
model = create_fraud_model('lightgbm')
model.fit(dataset_split.X_train, dataset_split.y_train, primary_label=PRIMARY_LABEL)
metrics = evaluate_fraud_model(
    model=model,
    X_valid=dataset_split.X_valid,
    y_valid=dataset_split.y_valid,
    X_test=dataset_split.X_test,
    y_test=dataset_split.y_test,
    primary_label=PRIMARY_LABEL,
)

print('Vio 主任务指标:')
print({k: v for k, v in metrics.items() if k != 'label_metrics'})
pd.DataFrame(metrics['label_metrics']).T[['accuracy', 'recall', 'f1', 'auc']]


/data/wk/code/github/automl_palyground/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data/wk/code/github/automl_palyground/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data/wk/code/github/automl_palyground/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data/wk/code/github/automl_palyground/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/data/wk/code/github/automl_palyground/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:27

Vio 主任务指标:
{'accuracy': 0.9022584504935687, 'recall': 0.5727040816326531, 'f1': 0.5787947147921366, 'auc': 0.8799504902350639, 'validation_metrics': {'accuracy': 0.904427161232426, 'recall': 0.5727040816326531, 'f1': 0.5842550422901757, 'auc': 0.8848381990885138}, 'decision_threshold': 0.92}


,accuracy,recall,f1,auc
Vio,0.902258,0.572704,0.578795,0.87995
V1,0.985343,0.495238,0.514851,0.925526
V2,0.998504,0.296296,0.444444,0.975438
V3,0.880272,0.711111,0.48964,0.884462
V4,0.89149,0.637375,0.441278,0.875542
V5,0.997083,0.073171,0.133333,0.894515


## 命令行训练

如果你希望直接通过项目入口训练，可以在仓库根目录执行：

```bash
uv run python src/runner.py --task fraud --algo lightgbm --fraud-split-folds 5 --fraud-fold-index 0 --fraud-train-ratio 0.7 --fraud-valid-ratio 0.1 --fraud-test-ratio 0.2
```
